1. Install Library

In [1]:
!pip install pdfplumber pandas numpy scikit-learn nltk openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 71.9 MB/s eta 0:00:00


2. Import Library

In [2]:
import os
import re
import glob
import pdfplumber
import pandas as pd
import numpy as np

3. Buat Struktur Folder

In [3]:
folders = [
    "data/raw",
    "data/processed",
    "data/eval",
    "data/results"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folder berhasil dibuat")

Folder berhasil dibuat


4. Upload PDF

In [4]:
from google.colab import files

uploaded = files.upload()

Saving putusan_4419_k_pid.sus_2020_20260605132123.pdf to putusan_4419_k_pid.sus_2020_20260605132123.pdf
Saving putusan_4009_k_pid.sus_2019_20260605132054.pdf to putusan_4009_k_pid.sus_2019_20260605132054.pdf
Saving putusan_1461_k_pid.sus_2019_20260605131929.pdf to putusan_1461_k_pid.sus_2019_20260605131929.pdf
Saving putusan_1519_k_pid.sus_2023_20260605131858.pdf to putusan_1519_k_pid.sus_2023_20260605131858.pdf
Saving putusan_186___pid.sus___2016___pn.sgr._20260605131821.pdf to putusan_186___pid.sus___2016___pn.sgr._20260605131821.pdf
Saving putusan_3583_k_pid.sus_2019_20260605131718.pdf to putusan_3583_k_pid.sus_2019_20260605131718.pdf
Saving putusan_21_pid_2018_pt.smr_20260605131458.pdf to putusan_21_pid_2018_pt.smr_20260605131458.pdf
Saving putusan_30_pid.sus_2018_pt_bbl_20260605131314.pdf to putusan_30_pid.sus_2018_pt_bbl_20260605131314.pdf
Saving putusan_995_k_pid.sus_2021_20260605131056 (1).pdf to putusan_995_k_pid.sus_2021_20260605131056 (1).pdf
Saving putusan_2835_k_pid.sus_20

5. Cek PDF

In [5]:
pdf_files = [f for f in os.listdir() if f.endswith(".pdf")]

print("Jumlah PDF:", len(pdf_files))

for file in pdf_files:
    print(file)

Jumlah PDF: 31
putusan_2001_k_pid.sus_2020_20260605125115.pdf
putusan_186___pid.sus___2016___pn.sgr._20260605131821.pdf
putusan_366_k_pid.sus_2021_20260605125023.pdf
putusan_671_k_pid.sus_2021_20260605130935.pdf
putusan_2051_k_pid.sus_2018_20260523212751.pdf
putusan_21_pid_2018_pt.smr_20260605131458.pdf
putusan_995_k_pid.sus_2021_20260605131056.pdf
putusan_30_pid.sus_2018_pt_bbl_20260605131314.pdf
putusan_2832_k_pid.sus_2018_20260605131109 (1).pdf
putusan_3583_k_pid.sus_2019_20260605131718.pdf
SubCPMK3 Genap 2025-2026.pdf
putusan_995_k_pid.sus_2021_20260605131056 (1).pdf
putusan_1519_k_pid.sus_2023_20260605131858.pdf
putusan_4311_k_pid.sus_2019_20260605125227.pdf
putusan_4009_k_pid.sus_2019_20260605132054.pdf
putusan_1461_k_pid.sus_2019_20260605131929.pdf
putusan_2835_k_pid.sus_2020_20260605131044 (1).pdf
putusan_2834_k_pid.sus_2020_20260605125103.pdf
putusan_2832_k_pid.sus_2018_20260605131109.pdf
putusan_2018_k_pid.sus_2020_20260605130847.pdf
putusan_1761_k_pid.sus_2019_20260605125239

6. Fungsi Ekstraksi PDF

In [6]:
def extract_text(pdf_path):

    text = ""

    with pdfplumber.open(pdf_path) as pdf:

        for page in pdf.pages:

            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

    return text

7. Fungsi Cleaning

In [7]:
def clean_text(text):

    text = text.lower()

    text = text.replace("\n", " ")

    text = re.sub(r'\b[a-zA-Z]\b', ' ', text)

    text = re.sub(
        r'putusan\.mahkamahagung\.go\.id',
        ' ',
        text
    )

    text = re.sub(
        r'direktori putusan mahkamah agung republik indonesia',
        ' ',
        text
    )

    text = re.sub(r'\s+', ' ', text)

    return text.strip()

8. Simpan ke data/raw

In [8]:
for i, pdf_file in enumerate(pdf_files):

    text = extract_text(pdf_file)

    cleaned = clean_text(text)

    with open(
        f"data/raw/case_{i+1:03d}.txt",
        "w",
        encoding="utf8"
    ) as f:

        f.write(cleaned)

print("Semua TXT berhasil dibuat")

Semua TXT berhasil dibuat


9. Buat Case Representation

In [9]:
cases = []

for i, pdf_file in enumerate(pdf_files):

    text = extract_text(pdf_file)

    cleaned = clean_text(text)

    cases.append({
        "case_id": f"C{i+1:03d}",
        "file_name": pdf_file,
        "text_full": cleaned
    })

df = pd.DataFrame(cases)

df.head()

,case_id,file_name,text_full
0,C001,putusan_2001_k_pid.sus_2020_20260605125115.pdf,gnomor 2001 /pid.sus/2020 demi keadilan berdas...
1,C002,putusan_186___pid.sus___2016___pn.sgr._2026060...,nomor : 186 / pid.sus / 2016 / pn.sgr. demi ke...
2,C003,putusan_366_k_pid.sus_2021_20260605125023.pdf,gnomor 366 /pid.sus/2021 demi keadilan berdasa...
3,C004,putusan_671_k_pid.sus_2021_20260605130935.pdf,gnomor 671 /pid.sus/2021 demi keadilan berdasa...
4,C005,putusan_2051_k_pid.sus_2018_20260523212751.pdf,nomor 2051 /pid.sus/2018 demi keadilan berdasa...


10. Simpan cases.csv

In [10]:
df.to_csv(
    "data/processed/cases.csv",
    index=False
)

print(df.shape)

(31, 3)


11. TF-IDF

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000
)

X = vectorizer.fit_transform(
    df["text_full"]
)

print(X.shape)

(31, 5000)


12. Cosine Similarity Retrieval

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

In [13]:
def retrieve(query, k=5):

    query = clean_text(query)

    query_vector = vectorizer.transform([query])

    similarity = cosine_similarity(
        query_vector,
        X
    )

    idx = similarity.argsort()[0][-k:][::-1]

    return df.iloc[idx]

13. Uji Retrieval

In [14]:
query = """
terdakwa memiliki sabu
dan melanggar pasal 112
"""

retrieve(query)

,case_id,file_name,text_full
9,C010,putusan_3583_k_pid.sus_2019_20260605131718.pdf,gnomor 3583 /pid.sus/2019 demi keadilan berdas...
25,C026,putusan_2279_k_pid.sus_2020_20260605125006.pdf,gnomor 2279 /pid.sus/2020 demi keadilan berdas...
16,C017,putusan_2835_k_pid.sus_2020_20260605131044 (1)...,gnomor 2835 /pid.sus/2020 demi keadilan berdas...
23,C024,putusan_2835_k_pid.sus_2020_20260605131044.pdf,gnomor 2835 /pid.sus/2020 demi keadilan berdas...
17,C018,putusan_2834_k_pid.sus_2020_20260605125103.pdf,gnomor 2834 /pid.sus/2020 demi keadilan berdas...


14. Tambahkan Label

In [16]:
def extract_label(text):

    text = text.lower()

    if "pasal 114" in text:
        return "Pasal114"

    elif "pasal 112" in text:
        return "Pasal112"

    elif "pasal 127" in text:
        return "Pasal127"

    else:
        return "Lainnya"

In [17]:
df["label"] = df["text_full"].apply(extract_label)

In [18]:
df[["case_id", "label"]].head(10)

,case_id,label
0,C001,Pasal112
1,C002,Pasal112
2,C003,Pasal112
3,C004,Pasal127
4,C005,Pasal114
5,C006,Pasal114
6,C007,Pasal114
7,C008,Pasal127
8,C009,Pasal114
9,C010,Pasal114


In [19]:
print(df["label"].value_counts())

label
Pasal114    22
Pasal112     5
Pasal127     3
Lainnya      1
Name: count, dtype: int64


15. Split Data

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    df["label"],
    test_size=0.2,
    random_state=42
)

16. SVM

In [22]:
from sklearn.svm import SVC

model = SVC(kernel="linear")

model.fit(X_train, y_train)

SVC(kernel='linear')

17. Prediksi

In [23]:
y_pred = model.predict(X_test)

18. Evaluasi

In [24]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

acc = accuracy_score(y_test, y_pred)

prec = precision_score(
    y_test,
    y_pred,
    average="weighted"
)

rec = recall_score(
    y_test,
    y_pred,
    average="weighted"
)

f1 = f1_score(
    y_test,
    y_pred,
    average="weighted"
)

print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1 Score :", f1)

Accuracy : 0.8571428571428571
Precision: 0.7346938775510203
Recall   : 0.8571428571428571
F1 Score : 0.7912087912087912


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [25]:
import pandas as pd

metrics = pd.DataFrame({
    "Accuracy": [acc],
    "Precision": [prec],
    "Recall": [rec],
    "F1 Score": [f1]
})

metrics.to_csv(
    "data/eval/retrieval_metrics.csv",
    index=False
)

metrics

,Accuracy,Precision,Recall,F1 Score
0,0.857143,0.734694,0.857143,0.791209


In [26]:
query1 = """
terdakwa memiliki sabu dan melanggar pasal 112
"""

retrieve(query1)

,case_id,file_name,text_full,label
9,C010,putusan_3583_k_pid.sus_2019_20260605131718.pdf,gnomor 3583 /pid.sus/2019 demi keadilan berdas...,Pasal114
25,C026,putusan_2279_k_pid.sus_2020_20260605125006.pdf,gnomor 2279 /pid.sus/2020 demi keadilan berdas...,Pasal114
16,C017,putusan_2835_k_pid.sus_2020_20260605131044 (1)...,gnomor 2835 /pid.sus/2020 demi keadilan berdas...,Pasal114
23,C024,putusan_2835_k_pid.sus_2020_20260605131044.pdf,gnomor 2835 /pid.sus/2020 demi keadilan berdas...,Pasal114
17,C018,putusan_2834_k_pid.sus_2020_20260605125103.pdf,gnomor 2834 /pid.sus/2020 demi keadilan berdas...,Pasal114


In [27]:
query2 = """
terdakwa menggunakan narkotika untuk diri sendiri
"""

retrieve(query2)

,case_id,file_name,text_full,label
9,C010,putusan_3583_k_pid.sus_2019_20260605131718.pdf,gnomor 3583 /pid.sus/2019 demi keadilan berdas...,Pasal114
28,C029,putusan_4078_k_pid.sus_2019_20260605125132.pdf,gnomor 4078 /pid.sus/2019 demi keadilan berdas...,Pasal114
4,C005,putusan_2051_k_pid.sus_2018_20260523212751.pdf,nomor 2051 /pid.sus/2018 demi keadilan berdasa...,Pasal114
7,C008,putusan_30_pid.sus_2018_pt_bbl_20260605131314.pdf,nnomor : 30 / pid.sus / 2018 / pt.bbl demi kea...,Pasal127
5,C006,putusan_21_pid_2018_pt.smr_20260605131458.pdf,ut an nomor 21/pid/2018/pt.smr demin keadilan ...,Pasal114


In [29]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve(query, k=5):

    query_vector = vectorizer.transform([query])

    similarity = cosine_similarity(
        query_vector,
        X
    )

    idx = similarity.argsort()[0][-k:][::-1]

    return df.iloc[idx]

In [30]:
from collections import Counter

def predict_outcome(query):

    top_cases = retrieve(query, 5)

    labels = list(top_cases["label"])

    prediction = Counter(labels).most_common(1)[0][0]

    return prediction

In [31]:
predict_outcome(
    "terdakwa memiliki sabu untuk diedarkan"
)

'Pasal114'